In [5]:
import torch
import math

# 1. Custom Autograd Implementations

class CustomReLU(torch.autograd.Function):
    """
    ReLU(x) = max(0, x)
    dReLU/dx = 1 if x > 0 else 0
    """
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        grad_x = grad_output.clone()
        grad_x[x <= 0] = 0
        return grad_x


class CustomSigmoid(torch.autograd.Function):
    """
    Sigmoid(x) = 1 / (1 + exp(-x))
    dSigmoid/dx = Sigmoid(x) * (1 - Sigmoid(x))
    """
    @staticmethod
    def forward(ctx, x):
        result = 1 / (1 + torch.exp(-x))
        ctx.save_for_backward(result)  # Save the output directly to save memory/compute
        return result

    @staticmethod
    def backward(ctx, grad_output):
        result, = ctx.saved_tensors
        grad_x = grad_output * result * (1.0 - result)
        return grad_x


class CustomTanh(torch.autograd.Function):
    """
    Tanh(x) = (exp(x) - exp(-x)) / (exp(x) + exp(-x))
    dTanh/dx = 1 - Tanh(x)^2
    """
    @staticmethod
    def forward(ctx, x):
        result = torch.tanh(x)
        ctx.save_for_backward(result)  # Save output to simplify backward pass
        return result

    @staticmethod
    def backward(ctx, grad_output):
        result, = ctx.saved_tensors
        grad_x = grad_output * (1.0 - result ** 2)
        return grad_x


class CustomGELU(torch.autograd.Function):
    """
    GELU(x) = x * P(X <= x) = 0.5 * x * (1 + erf(x / sqrt(2)))
    Using the exact formulation here (via torch.erf).
    dGELU/dx = 0.5 * (1 + erf(x / sqrt(2))) + (x / sqrt(2 * pi)) * exp(-x^2 / 2)
    """
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        # 0.5 * x * (1 + erf(x / sqrt(2)))
        return 0.5 * x * (1.0 + torch.erf(x / math.sqrt(2.0)))

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        cdf = 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))
        pdf = torch.exp(-0.5 * x**2) / math.sqrt(2.0 * math.pi)
        grad_x = grad_output * (cdf + x * pdf)
        return grad_x


class CustomSwish(torch.autograd.Function):
    """
    Swish(x) = x * Sigmoid(x)
    dSwish/dx = Sigmoid(x) + x * Sigmoid(x) * (1 - Sigmoid(x))
              = Swish(x) + Sigmoid(x) * (1 - Swish(x))
    """
    @staticmethod
    def forward(ctx, x):
        sigmoid_x = 1 / (1 + torch.exp(-x))
        result = x * sigmoid_x
        ctx.save_for_backward(x, sigmoid_x)
        return result

    @staticmethod
    def backward(ctx, grad_output):
        x, sigmoid_x = ctx.saved_tensors
        grad_x = grad_output * (sigmoid_x + x * sigmoid_x * (1.0 - sigmoid_x))
        return grad_x

# Helper functions to call them cleanly
relu_custom = CustomReLU.apply
sigmoid_custom = CustomSigmoid.apply
tanh_custom = CustomTanh.apply
gelu_custom = CustomGELU.apply
swish_custom = CustomSwish.apply


In [4]:
def run_gradient_unit_tests():
    # Generate random high-precision tensors for testing
    # Avoid zero for ReLU to prevent analytical/numerical disagreement at the exact inflection point
    torch.manual_seed(42)
    test_input = torch.randn(3, 3, dtype=torch.float64, requires_grad=True)

    print("--- Running Autograd Gradcheck Unit Tests ---")
    
    # 1. Test ReLU
    relu_input = test_input.clone().detach().requires_grad_(True)
    assert torch.autograd.gradcheck(relu_custom, relu_input, eps=1e-6, atol=1e-4), "ReLU gradcheck failed"
    print(" Custom ReLU gradient verified successfully.")

    # 2. Test Sigmoid
    assert torch.autograd.gradcheck(sigmoid_custom, test_input, eps=1e-6, atol=1e-4), "Sigmoid gradcheck failed"
    print(" Custom Sigmoid gradient verified successfully.")

    # 3. Test Tanh
    assert torch.autograd.gradcheck(tanh_custom, test_input, eps=1e-6, atol=1e-4), "Tanh gradcheck failed"
    print(" Custom Tanh gradient verified successfully.")

    # 4. Test GELU
    assert torch.autograd.gradcheck(gelu_custom, test_input, eps=1e-6, atol=1e-4), "GELU gradcheck failed"
    print(" Custom GELU gradient verified successfully.")

    # 5. Test Swish
    assert torch.autograd.gradcheck(swish_custom, test_input, eps=1e-6, atol=1e-4), "Swish gradcheck failed"
    print(" Custom Swish gradient verified successfully.")
    
    print("\nAll unit tests passed. The backward functions are mathematically correct.")

# Run the test suite
run_gradient_unit_tests()


--- Running Autograd Gradcheck Unit Tests ---
 Custom ReLU gradient verified successfully.
 Custom Sigmoid gradient verified successfully.
 Custom Tanh gradient verified successfully.
 Custom GELU gradient verified successfully.
 Custom Swish gradient verified successfully.

All unit tests passed. The backward functions are mathematically correct.
